In [16]:
!ls -lah /content

ls: /content: No such file or directory


In [24]:
import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from torchsummary import summary

import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

PROJECT_PATH = Path.cwd().parent
DATA_DIR = PROJECT_PATH / 'data'
MODEL_DIR = PROJECT_PATH / 'model' / 'lenet_model.pt'
DATA_DIR

PosixPath('/Users/khengngok/PycharmProjects/AIOExercies/computer_vision_architectures_module/CNN/cassava_leaf_disease_classifier_streamlit/data')

In [27]:
data_paths = {
    'train': DATA_DIR / 'train',
    'valid': DATA_DIR / 'validation',
    'test': DATA_DIR / 'test',
}

def loader(path):
    return Image.open(path)

img_size = 150
train_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size))
    transforms.ToTensor(),
])

train_data = datasets.ImageFolder(
    root=data_paths['train'],
    loader=loader,
    transform=train_transforms
)
valid_data = datasets.ImageFolder(
    root=data_paths['valid']
    transform=train_transforms
)
test_data = datasets.ImageFolder(
    root=data_paths['test'],
    transform=train_transforms
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3763838872.py, line 12)

In [22]:
class LeNetClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=6,
            kernel_size=5,
            padding='same'
        )
        self.avgpool1 = nn.AvgPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(
            in_channels=6,
            out_channels=16,
            kernel_size=5
        )
        self.avgpool2 = nn.AvgPool2d(kernel_size=2)
        self.flattern = nn.Flattern()
        self.fc_1 = nn.Linear(16 * 35 * 35, 120)
        self.fc_2 = nn.Linear(120, 84)
        self.fc_3 = nn.Linear(84, num_classes)

    def forward(self, inputs):
        outputs = self.conv1(inputs)
        outputs = self.avgpool1(outputs)
        outputs = F.relu(outputs)
        outputs = self.conv2(outputs)
        outputs = self.avgpool2(outputs)
        outputs = F.relu(outputs)
        outputs = self.flatten(outputs)
        outputs = self.fc_1(outputs)
        outputs = self.fc_2(outputs)
        outputs = self.fc_3(outputs)

        return outputs

In [23]:
import time
def train(model, criterion, optimizer, train_dataloader, device, epoch=0,
          log_interval=50):
    model.train()
    total_acc, total_count = 0, 0
    losses = []
    start_time = time.time()

    for idx, (inputs, labels) in enumerate(train_dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        predictions = model(inputs)

        loss = criterion(predictions, labels)
        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)

        optimizer.step()

        total_acc += (redictions.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start.time()
            print("-" * 60)
            print("| End of epoch {:3d} | Time : {:5.2f}s | Train Accuracy {:8.3f} | Train Loss {:8.3f} "
                "| Valid Accuracy {:8.3f} | Valid Loss {:8.3f} ".format(
                    epoch, time.time() - epoch_start_time, train_acc, train_loss,
                    eval_acc, eval_loss
                )
            )
            print("-" * 60)

            total_acc, total_count = 0, 0
            start_time = time.time()


    epoch_acc = total_acc / total_count
    epoch_loss = sum(losses) / len(losses)

    return epoch_acc, epoch_loss

def evaluate(model, criterion, valid_dataloader):
    model.eval()

    total_acc, total_count = 0, 0
    losses = []

    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(valid_dataloader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            predictions = model(inputs)

            loss = criterion(predictions, labels)

            losses.append(loss.item())

            total_acc += (predictions.argmax(1) == labels).sum().item()
            total_count += labels.size(0)

    epoch_acc = total_acc / total_count
    epoch_loss = sum(losses) / len(losses)

    return epoch_acc, epoch_loss

In [28]:
num_classes = len(train_data.classes)

lenet_model = LeNetClassifier(num_classes)
lenet_model.to(device)

criterion = torch.nn.CrossEntropyLoss()
learning_rate = 2e-4
optimizer = optim.Adam(lenet_model.parameter(),
                       learning_rate)

num_epochs = 10

train_accs, train_losses = [], []
eval_acc, eval_losses = [], []
best_loss_eval = 100

for epoch in range (1, num_epochs+1):
    epoch_start_time = time.time()

    train_acc, train_loss = train(model, criterion, optimizer, train_dataloader,
    device, epoch, log_iterval=10)
    train_accs.append(train_acc)
    train_losses.append(train_loss)

    eval_acc, eval_loss = evaluate(model, criterion, valid_dataloader)
    eval_accs.append(eval_acc)
    eval_losses.append(eval_loss)

    if eval_loss < best_loss_eval:
        torch.save(lenet_model.state_dict(), MODEL_DIR)

    print("-" * 60)
    print("| End of epoch {:3d} | Time : {:5.2f}s | Train Accuracy {:8.3f} | Train Loss {:8.3f} "
        "| Valid Accuracy {:8.3f} | Valid Loss {:8.3f} ".format(
            epoch, time.time() - epoch_start_time, train_acc, train_loss,
            eval_acc, eval_loss
        )
    )
    print("-" * 60)

    lenet_model.load_state_dict(torch.load(MODEL_DIR))
    lenet_model.eval()

NameError: name 'train_data' is not defined

In [29]:
test_acc, test_loss = evaluate(lenet_model, criterion, test_dataloader)
test_acc, test_loss

NameError: name 'lenet_model' is not defined